# Milestone 5 lab, notebook 6 of 7: Every five minutes

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| **6. Every five minutes** (you are here) | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

**This notebook is a lab.** The daily report does not need anything in it.

In milestone 1 you chose a daily batch, because the manager reads the report once a morning. You also met two other shapes by name: micro-batch and streaming. Now that the pipeline is on a scheduler, you can try the first one. It takes one changed line.

In [1]:
import pandas as pd

from lab_helpers import MILESTONE_DIR, api, arden_time, at, show_doc, table

at.wait_until_ready()

## Step 1: The same pipeline, a smaller slice

Here is the lab's DAG. Compare it with the daily one.

In [2]:
print((MILESTONE_DIR / "dags" / "orders_every_five_minutes_dag.py").read_text())

"""A lab: the same pipeline as a micro-batch. It runs every five minutes.

It starts paused. Nothing here is needed for the daily report.
It writes to its own database, so it cannot disturb the daily DAG.
"""

from datetime import datetime

from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import DAG

PYTHON = "/opt/arden-venv/bin/python"
# Its own database, and no lookback: with a five-minute slice, each run should read five minutes.
PIPELINE = (
    "cd /opt/arden/src/m5_on_a_schedule && "
    f"ARDEN_DATABASE=arden_m5_micro ARDEN_LOOKBACK_DAYS=0 {PYTHON} -m daily_orders"
)

with DAG(
    dag_id="orders_every_five_minutes",
    description="Lab: the same batch, with a five-minute slice.",
    schedule="*/5 * * * *",  # every five minutes
    start_date=datetime(2026, 1, 1),
    catchup=False,
    max_active_runs=1,
    is_paused_upon_creation=True,
    tags=["arden", "lab"],
) as dag:
    # Five minutes pass in the fake order system, as they do on th

The tasks run the **same program**: `extract-load`, then `transform-test`. Two things differ.

- The schedule is `*/5 * * * *`: every five minutes.
- The first task lets five minutes pass in the fake order system, as five minutes pass on the wall.

It writes to its own database, `arden_m5_micro`, so it cannot disturb the daily DAG. And it starts switched off.

## Step 2: Run an hour of slices

Set the order system to a busy moment: Friday 12 June at noon. Then run the DAG thirteen times: one first run, and twelve five-minute slices, which is one hour. `airflow dags test` runs one whole run at once, so you do not have to wait five real minutes between them. This cell takes about two minutes.

In [3]:
from daily_orders import warehouse

warehouse.drop_database("arden_m5_micro")
api.post("/admin/clock", json={"now": "2026-06-12T17:00:00Z"})        # Friday 12:00 in Arden

picked_up_at = {}
for number in range(1, 14):
    at.airflow("dags", "test", "orders_every_five_minutes")
    picked_up_at[f"2026-06-12_{number}"] = api.get("/admin/clock").json()["now_utc"]

print("The order system's clock is now", api.get("/admin/clock").json()["now_local"][11:16], "in Arden.")

The order system's clock is now 13:05 in Arden.


Thirteen runs, and the order system's clock moved five minutes each time. Look at what each run loaded.

In [4]:
con = warehouse.connect("arden_m5_micro")
runs = table(con, "SELECT run_id, rows_landed, api_calls, tests_passed FROM ops.load_runs ORDER BY loaded_at")
runs

,run_id,rows_landed,api_calls,tests_passed
0,2026-06-12_1,353,16,31
1,2026-06-12_2,2,1,31
2,2026-06-12_3,1,1,31
3,2026-06-12_4,1,1,31
4,2026-06-12_5,3,1,31
5,2026-06-12_6,1,1,31
6,2026-06-12_7,1,1,31
7,2026-06-12_8,2,1,31
8,2026-06-12_9,2,1,31
9,2026-06-12_10,2,1,31


The first run had no watermark, so it read everything. After that, each run read **one or two rows**: what changed in its five minutes, plus the boundary order that is always read again.

Each of these is a normal batch run. It extracts, loads, builds the models, and passes all 31 table tests. Nothing about it is "streaming".

## Step 3: Measure the latency

Latency is the time from a change in the source until the warehouse can show it. List every change the twelve slices picked up: when it happened, and when a run loaded it.

In [5]:
rows = []
for run_id in runs["run_id"][1:]:
    picked_up = arden_time(picked_up_at[run_id])
    changes = con.execute(
        "SELECT (payload->>'order_id')::int, payload->>'status', payload->>'updated_at' FROM raw.orders_landing WHERE run_id = %s ORDER BY 3",
        [run_id],
    ).fetchall()
    for order_id, status, updated_at in changes[1:]:              # [1:] skips the boundary order
        happened = arden_time(updated_at)
        rows.append({"order": order_id, "became": status, "happened at": happened.strftime("%H:%M:%S"),
                     "loaded at": picked_up.strftime("%H:%M"), "waited (minutes)": round((picked_up - happened).total_seconds() / 60, 1)})

waits = pd.DataFrame(rows)
waits

,order,became,happened at,loaded at,waited (minutes)
0,1354,placed,12:08:07,12:10,1.9
1,1352,out_for_delivery,12:22:52,12:25,2.1
2,1351,delivered,12:24:22,12:25,0.6
3,1353,out_for_delivery,12:35:46,12:40,4.2
4,1354,packed,12:43:07,12:45,1.9
5,1355,placed,12:46:24,12:50,3.6


In [6]:
print(f"{len(waits)} changes in one hour. The longest wait was {waits['waited (minutes)'].max()} minutes.")

6 changes in one hour. The longest wait was 4.2 minutes.


No change waited longer than five minutes. Compare that with the daily batch of milestone 1, where the refund on order 1130 waited **20 hours**.

| | Daily batch | This micro-batch |
|---|---|---|
| The slice | One day | Five minutes |
| Longest wait for a change | Almost a day | Five minutes, plus the time a run takes |
| Runs a day | 1 | 288 |
| Calls to the order system a day | A handful | At least 288 |

Most of the twelve runs found nothing new, or one change. They still ran.

## Step 4: How far can you push it?

Change `*/5` to `*/1`, and the wait is one minute. Why not one second?

Look at what one run costs. It starts a program, asks the API, opens the warehouse, builds seven models and runs 31 tests. On this computer one run of the lab DAG takes a few seconds, however little data there is.

- At a five-minute slice, a few seconds is nothing.
- At a one-minute slice, it is a visible part of the wait.
- At a one-second slice, the run takes longer than the slice. The runs pile up.

A micro-batch can never be faster than one run of the batch. Below that, you need a different kind of system: a job that never stops, and handles each change as it arrives. That is **streaming**.

| Shape | How it runs | Latency | What it costs you |
|---|---|---|---|
| **Batch** | One run for a whole slice, then it stops | Up to one slice | Least. A failed run is run again |
| **Micro-batch** | The same batch, with a very small slice | Up to one slice, plus the run time | More runs, more calls to the source |
| **Streaming** | A job that never stops | A short moment | Most. You manage state, late events and restarts |

Streaming is **not built** in this project. It belongs to the clickstream project of the series. What you can take from this lab: micro-batch is not a third kind of system. It is a batch with a small slice, and everything you built, the watermark, the MERGE idea, the tests, still does its job.

Does Arden need it? No. The manager reads the report once, at 08:00. A five-minute pipeline would call the order system 288 times a day to produce a report that is read once.

In [7]:
con.close()
show_doc("HLD.md", "8. The micro-batch lab")

**From `docs/HLD.md`**

> ## 8. The micro-batch lab
> 
> A second, small DAG runs the **same pipeline every five minutes**. It is a lab, and it starts switched off.
> 
> | | Daily batch | Micro-batch (the lab) | Streaming (not built) |
> |---|---|---|---|
> | The slice | One day | Five minutes | One event |
> | Latency | Up to one day | Up to five minutes, plus the run time | A short moment |
> | What you run | This DAG | The same tasks, a different schedule | A different kind of system |
> 
> It shows how far a batch can be pushed. Each run is still a normal batch run. Below about a minute, the time a run takes is most of the wait, and a stream becomes the better tool.

To see the lab run on its own schedule, switch it on in Airflow's page at http://localhost:8080 , watch a run appear every five minutes, and switch it off again. Or from a terminal:

```
docker compose exec airflow airflow dags unpause orders_every_five_minutes
docker compose exec airflow airflow dags pause orders_every_five_minutes
```

## A word, now with a measurement

| Word | Meaning | Here |
|---|---|---|
| **Micro-batch** | "Micro-batch is batch with a very small slice. The job runs again and again, for example every minute. Each run is a normal batch run." | The same pipeline every five minutes: no change waited more than five minutes |

The definition is from the D8Loop chapter [Batch vs Streaming vs Micro-Batch](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/batch-vs-streaming-vs-micro-batch).

## What you now know

- A micro-batch is the same batch pipeline on a shorter schedule. One line changed.
- Its latency is up to one slice, plus the time one run takes.
- A batch cannot be pushed below the time of one run. Past that point a stream is the better tool.
- The right latency is the one the reader needs. Arden's report needs one day.

**Say it in an interview:** "I ran my daily pipeline as a micro-batch every five minutes in a lab. Latency fell from up to a day to under five minutes, with the same code and the same tests. The limit is the run time: each run costs a few seconds whatever the data, so below about a minute I would move to streaming. For a report read once a morning, daily batch was the right choice."

Next: [notebook 7](07_the_bad_morning.ipynb) is the final drill.